In [5]:
import random
import torch
import torch.nn
from data.TripletsDataset import TripletsDataset

NumExpr will use 8 threads


In [12]:
import ctypes
import os
lib = ctypes.CDLL('base/libexample.so')

result = lib.add(5, 4)
print(f"The result is: {result}")

The result is: 9


In [12]:

dataset = TripletsDataset('datasets/nations')
dataset

In [2]:


def generate_negative_samples(positive_triples, num_entities, num_samples):
    # Create dictionaries to store head and tail sets for each relation
    head_dict = {}
    tail_dict = {}
    for (h, r, t) in positive_triples:
        if r not in head_dict:
            head_dict[r] = set()
        if r not in tail_dict:
            tail_dict[r] = set()
        head_dict[r].add(h)
        tail_dict[r].add(t)

    negative_samples = []
    for (h, r, t) in positive_triples:
        heads = list(head_dict[r])
        tails = list(tail_dict[r])

        # Generate negative samples by corrupting heads
        for _ in range(num_samples):
            corrupted_h = random.choice(heads)
            if (corrupted_h, r, t) not in positive_triples:
                negative_samples.append((corrupted_h, r, t))

        # Generate negative samples by corrupting tails
        for _ in range(num_samples):
            corrupted_t = random.choice(tails)
            if (h, r, corrupted_t) not in positive_triples:
                negative_samples.append((h, r, corrupted_t))

    return negative_samples

# Example usage
positive_triples = [('Alice', 'friendOf', 'Bob'), 
                    ('Alice', 'friendOf', 'Carol'), 
                    ('David', 'friendOf', 'Bob')]

num_entities = 4  # Just an example, normally you would have the actual number of entities
num_samples = 1  # Number of negative samples per positive triple

negative_samples = generate_negative_samples(positive_triples, num_entities, num_samples)
print("Negative Samples:", negative_samples)


Negative Samples: []


In [18]:
positive_scores = torch.tensor([3.0, 2.0])
negative_scores = torch.tensor([1.0, 1.5, 2.5, 3.0])
 
negative_scores = negative_scores.view(-1, len(positive_scores))
negative_scores = negative_scores.permute(1, 0)


print(positive_scores.unsqueeze(1) )
print(negative_scores)
print(positive_scores.unsqueeze(1)  - negative_scores)
target = torch.tensor([[1., 1.], [1., 1.]])

criterion = torch.nn.MarginRankingLoss(margin=1.0)
loss = criterion(positive_scores.unsqueeze(1), negative_scores, target)

print(loss)

tensor([[3.],
        [2.]])
tensor([[1.0000, 2.5000],
        [1.5000, 3.0000]])
tensor([[ 2.0000,  0.5000],
        [ 0.5000, -1.0000]])
tensor(0.7500)
